In [1]:
#test

In [ ]:
import datetime
import logging  # ログ制御用
import warnings  # 警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Node.js Theme Palette)
# =====================================================================
def show_market_performances(market_dict, title="Market Performance Overview"):
    today = datetime.date.today()
    # 1年分（約370日）のデータを取得して過去騰落率を正確に計算
    start_date = today - datetime.timedelta(days=370)
    
    # Node.js テーマ風のタイトルヘッダー (Dark Background & Neon Green Text)
    title_html = f"""
    <div style='
        background-color: #181818; 
        border-left: 5px solid #68a063; 
        padding: 12px 18px; 
        border-radius: 4px; 
        margin-bottom: 25px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    '>
        <h2 style='
            margin: 0; 
            color: #539e43; 
            font-family: "Consolas", "Courier New", Monaco, monospace; 
            font-size: 22px;
            letter-spacing: 0.5px;
        '>⬢ {title}</h2>
        <span style='
            font-size: 12px; 
            color: #888888; 
            font-family: monospace;
        '>System Status: ONLINE | As of {today.strftime('%Y-%m-%d')}</span>
    </div>
    """
    display(HTML(title_html))
    
    # 数値フォーマット設定
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # Node.jsテーマに合わせたハイライト関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        # プラス（上昇）：Node.jsのネオングリーン背景
        if val >= 10:
            return 'background-color: #1e3a1e; color: #41d888; font-weight: bold; border-radius: 2px;'
        # マイナス（下落）：ダークレッド/コーラル系
        elif val <= -10:
            return 'background-color: #3b1818; color: #ff6b6b; font-weight: bold; border-radius: 2px;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # セクションヘッダー（Node Dark Gray + Green Accent）
        section_html = f"""
        <div style='
            margin-top: 20px; 
            margin-bottom: 8px; 
            color: #215732; 
            font-family: "Consolas", monospace; 
            font-weight: bold;
            font-size: 15px;
            border-bottom: 1px solid #333333;
            padding-bottom: 4px;
        '>
            <span style='color: #68a063;'>❯</span> {market_name}
        </div>
        """
        display(HTML(section_html))
        
        for name, ticker in ticker_dict.items():
            try:
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # テーブル全体のスタイル定義（Dark Node UI Style）
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .set_table_styles([
                             # テーブル全体
                             {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', 'Consolas, monospace'), ('font-size', '13px'), ('background-color', '#121212'), ('color', '#e0e0e0')]},
                             # ヘッダー行（Dark Gray / Green Accent）
                             {'selector': 'th', 'props': [('background-color', '#1f1f1f'), ('color', '#68a063'), ('border', '1px solid #333'), ('padding', '8px 12px'), ('text-align', 'center')]},
                             # データセル
                             {'selector': 'td', 'props': [('border', '1px solid #282828'), ('padding', '7px 12px'), ('text-align', 'right')]},
                             # ホバー効果
                             {'selector': 'tr:hover', 'props': [('background-color', '#1a261a')]}
                         ])
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [ ]:
MARKET_TICKERS = {
    # ==========================================
    # 1. アメリカ市場（米国株・指数・ETF）
    # ==========================================
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG",
        "Vanguard FTSE Global All-Cap UCITS ETF (VALL.L)": "VALL.L",
    },

    # ==========================================
    # 2. 日本市場（日本株・指数）
    # ==========================================
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Japan Stock (Custom)": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },

    # ==========================================
    # 3. シンガポール市場（総合・インデックス）
    # ==========================================
    "Singapore (General)": {
        "STI Index": "^STI",
        "DBS Group (金融要塞/東南アジア最大手)": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
        "SGX Group (完全独占/国内唯一の証券取引所)": "S68.SI",
        "PropNex (不動産仲介最大手/高配当)": "OYY.SI",
    },

    # ==========================================
    # 4. シンガポール：医療セクター (Healthcare)
    # ==========================================
    "Singapore Healthcare": {
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone Holdings (クリーンルーム用手袋)": "AP4.SI",
    },

    # ==========================================
    # 5. シンガポール：食品・アグリビジネス
    # ==========================================
    "Singapore Food & Agribusiness": {
        "Wilmar International (世界最大級のパーム油)": "F34.SI",
        "Olam Group (ココア・コーヒー・ナッツ等大手)": "VC2.SI",
    },

    # ==========================================
    # 6. シンガポール：コア・安定・高配当・バリュー
    # ==========================================
    "Singapore Core & Stability": {
        "Sembcorp Industries (大手公益/エネルギー)": "U96.SI",
        "Valuetronics Holdings (電子機器製造受託/無借金)": "BN2.SI",
    },
    "Singapore High Yield & Growth": {
        "Multi-Chem (ITセキュリティ/高配当)": "AWZ.SI",
        "Yangzijiang Financial Holding (金融)": "YF8.SI",
        "Yangzijiang Shipbuilding (造船/高収益)": "BS6.SI",
        "Sanli Environmental Limited": "1E3.SI",
    },
    "Singapore Recovery & Value": {
        "Micro-Mechanics (無借金/半導体消耗工具/高配当)": "508.SI",
    },

    # ==========================================
    # 7. シンガポール：建設・インフラ
    # ==========================================
    "Singapore Construction & Infrastructure": {
        "BRC Asia (鉄筋加工最大手)": "BEC.SI",
        "OKP Holdings (道路土木インフラ/無借金)": "5CF.SI",
        "ISOTeam (HDB建物修繕塗装R&R)": "5WF.SI",
    },
}



In [ ]:
show_market_performances(MARKET_TICKERS, title="Market Performance Overview")

# Gemini Prompt
以下の【分析データ】を基に、指定の【Pythonコードフォーマット】に変換して出力してください。

【出力条件】
・MARKET_TICKERS の辞書形式で出力すること。
・辞書のキーには「企業名 (ティッカー・主要特徴/指標の要約)」を含めること。
・値（Value）には「ティッカーシンボル（.SI付き）」を入れること。
・解説文や導入の挨拶は不要で、そのままコピーして使えるPythonコードのみを出力すること。

【Pythonコードフォーマット例】
MARKET_TICKERS = {
    "カテゴリ名": {
        "銘柄名 (ティッカー・特徴要約)": "ティッカー.SI",
    },
}

【分析データ】
（※ここに分析したテキストや銘柄リスト、あるいは「以下の銘柄を分析してコード化して」と入力）